In [96]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [97]:
dataset=pd.read_csv("CKD.csv")

In [98]:
dataset.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.0,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.0,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.0,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.0,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.0,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes


In [99]:
dataset.isnull().sum()

age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hrmo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

In [100]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [101]:
dataset=dataset.astype(int)

In [102]:
dataset.head()

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2,76,3,0,148,57,3,137,4,12,...,0,0,0,0,0,0,1,1,0,1
1,3,76,2,0,148,22,0,137,4,10,...,1,0,0,0,0,0,1,0,0,1
2,4,76,1,0,99,23,0,138,4,12,...,1,0,0,0,0,0,1,0,0,1
3,5,76,1,0,148,16,0,138,3,8,...,1,0,0,0,0,0,1,0,1,1
4,5,50,0,0,148,25,0,137,4,11,...,1,0,0,0,0,0,1,0,0,1


In [103]:
indep= dataset.iloc[:,0:27].values
dep=dataset['classification_yes']

In [104]:
indep

array([[ 2, 76,  3, ...,  1,  1,  0],
       [ 3, 76,  2, ...,  1,  0,  0],
       [ 4, 76,  1, ...,  1,  0,  0],
       ...,
       [51, 70,  3, ...,  0,  0,  0],
       [51, 90,  0, ...,  1,  0,  1],
       [51, 80,  0, ...,  1,  0,  0]])

In [105]:
dep

0      1
1      1
2      1
3      1
4      1
      ..
394    1
395    1
396    1
397    1
398    0
Name: classification_yes, Length: 399, dtype: int32

In [106]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(indep,dep,test_size=1/3,random_state=0)

In [107]:
x_train,x_test,y_train,y_test

(array([[ 60, 100,   0, ...,   1,   0,   0],
        [ 57,  80,   0, ...,   1,   0,   0],
        [ 42,  70,   0, ...,   1,   0,   0],
        ...,
        [ 45,  80,   0, ...,   1,   0,   0],
        [ 30,  80,   0, ...,   1,   0,   0],
        [ 51,  60,   0, ...,   1,   0,   0]]),
 array([[ 46,  70,   0, ...,   1,   0,   0],
        [ 66,  90,   2, ...,   0,   0,   0],
        [ 69,  70,   0, ...,   1,   0,   0],
        ...,
        [ 48, 100,   0, ...,   0,   0,   0],
        [ 51,  60,   3, ...,   1,   0,   0],
        [ 34,  60,   0, ...,   1,   0,   0]]),
 240    1
 218    1
 101    0
 311    1
 194    1
       ..
 323    1
 192    1
 117    1
 47     0
 172    0
 Name: classification_yes, Length: 266, dtype: int32,
 132    0
 309    1
 334    0
 196    1
 246    1
       ..
 349    0
 168    0
 150    1
 392    1
 66     0
 Name: classification_yes, Length: 133, dtype: int32)

In [108]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [109]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {'solver':['newton-cg', 'lbfgs', 'liblinear', 'saga'],
             'penalty':['l2']} 



grid = GridSearchCV(LogisticRegression(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 

In [110]:
grid.fit(x_train,y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


GridSearchCV(estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'penalty': ['l2'],
                         'solver': ['newton-cg', 'lbfgs', 'liblinear', 'saga']},
             scoring='f1_weighted', verbose=3)

In [111]:
re=grid.cv_results_

In [112]:
grid_predictions=grid.predict(x_test)

In [113]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,grid_predictions)

In [117]:
print(cm)


[[51  0]
 [ 1 81]]


In [118]:
from sklearn.metrics import classification_report
clf_report=classification_report(y_test,grid_predictions)

In [119]:
print(clf_report)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99        51
           1       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



In [120]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'penalty': 'l2', 'solver': 'newton-cg'}: 0.9924946382275899


In [123]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])


1.0

In [124]:
table=pd.DataFrame.from_dict(re)

In [125]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.009175,0.000978,0.003989,0.000631,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.981569,0.962264,0.981217,1.000000,1.000000,0.985010,0.014093,1
1,0.005785,0.000399,0.003789,0.000746,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.981569,0.962264,0.981217,1.000000,1.000000,0.985010,0.014093,1
2,0.002792,0.000398,0.004389,0.000798,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.963284,0.981233,0.962573,0.981217,0.981217,0.973905,0.008965,4
3,0.019747,0.003116,0.003590,0.000488,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.981569,0.981233,0.981217,0.981217,0.981217,0.981291,0.000139,3


In [126]:
#Future prediction

In [127]:
import pickle

In [128]:
filename="finalized_model_LogisticRegression_GridClassification.sav"

In [129]:
pickle.dump(grid ,open(filename,'wb'))

In [130]:
#Standardscaler

In [131]:
dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [132]:
dataset.head()

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2,76,3,0,148,57,3,137,4,12,...,0,0,0,0,0,0,1,1,0,1
1,3,76,2,0,148,22,0,137,4,10,...,1,0,0,0,0,0,1,0,0,1
2,4,76,1,0,99,23,0,138,4,12,...,1,0,0,0,0,0,1,0,0,1
3,5,76,1,0,148,16,0,138,3,8,...,1,0,0,0,0,0,1,0,1,1
4,5,50,0,0,148,25,0,137,4,11,...,1,0,0,0,0,0,1,0,0,1


In [133]:
preinput=sc.transform([[6,82,1,95,17,0,137,3,12,38,7500,5.4,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1]])

In [134]:
loaded_model=pickle.load(open("finalized_model_LogisticRegression_GridClassification.sav",'rb'))

In [135]:
result=loaded_model.predict(preinput)

In [136]:
result

array([0])